<a href="https://colab.research.google.com/github/MuhammadRayan07/Weather-Analysis/blob/main/notebooks/02_Analysis_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BASE = Path("/content/drive/MyDrive/weather-analysis")
PROCESSED = BASE / "data" / "processed"

weather = pd.read_csv(PROCESSED / "weather_combined.csv", parse_dates=["date", "collected_at"])
print(weather.shape)

Mounted at /content/drive
(73060, 24)


In [2]:
SEED = 15
np.random.seed(SEED)
results_log = []

# 4. Dataset Understanding and Description

## 4.1

In [3]:
weather.head()


,city,country,latitude,longitude,date,weather_code,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,...,wind_direction_10m_dominant,shortwave_radiation_sum,sunshine_duration,daylight_duration,relative_humidity_2m_mean,dew_point_2m_mean,cloud_cover_mean,pressure_msl_mean,surface_pressure_mean,collected_at
0,Lahore,Pakistan,31.5497,74.3436,2016-01-01,3,20.9,8.9,14.5,0.0,...,340,10.82,35639.67,36470.50,64,7.2,46,1021.0,994.5,2026-11-22 14:39:52+00:00
1,Lahore,Pakistan,31.5497,74.3436,2016-01-02,3,20.6,6.2,13.0,0.0,...,351,12.30,35442.13,36500.41,69,6.8,37,1021.9,995.3,2026-11-22 14:39:52+00:00
2,Lahore,Pakistan,31.5497,74.3436,2016-01-03,3,21.6,7.5,14.1,0.0,...,298,12.20,35613.56,36532.97,65,6.9,34,1018.7,992.3,2026-11-22 14:39:52+00:00
3,Lahore,Pakistan,31.5497,74.3436,2016-01-04,51,20.9,9.5,14.6,0.6,...,99,9.30,28133.52,36568.10,73,9.5,63,1018.7,992.3,2026-11-22 14:39:52+00:00
4,Lahore,Pakistan,31.5497,74.3436,2016-01-05,3,21.9,8.8,14.8,0.0,...,354,12.77,35593.52,36605.75,77,10.3,27,1018.6,992.2,2026-11-22 14:39:52+00:00


In [4]:
weather.tail()

,city,country,latitude,longitude,date,weather_code,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,...,wind_direction_10m_dominant,shortwave_radiation_sum,sunshine_duration,daylight_duration,relative_humidity_2m_mean,dew_point_2m_mean,cloud_cover_mean,pressure_msl_mean,surface_pressure_mean,collected_at
73055,Sao Paulo,Brazil,-23.5505,-46.6333,2025-12-27,63,35.6,22.6,27.1,4.4,...,356,26.75,47943.35,48864.91,66,19.7,51,1013.6,930.5,2026-11-22 15:03:58+00:00
73056,Sao Paulo,Brazil,-23.5505,-46.6333,2025-12-28,65,35.0,23.1,27.9,13.2,...,208,27.64,48255.21,48852.27,61,18.9,48,1012.0,929.2,2026-11-22 15:03:58+00:00
73057,Sao Paulo,Brazil,-23.5505,-46.6333,2025-12-29,63,33.6,22.6,26.2,5.0,...,344,27.59,46254.16,48837.74,71,19.9,69,1012.0,928.8,2026-11-22 15:03:58+00:00
73058,Sao Paulo,Brazil,-23.5505,-46.6333,2025-12-30,63,31.4,20.9,24.9,8.0,...,350,27.12,46018.42,48821.30,78,20.5,71,1013.0,929.3,2026-11-22 15:03:58+00:00
73059,Sao Paulo,Brazil,-23.5505,-46.6333,2025-12-31,63,29.7,20.1,23.9,8.0,...,320,22.06,37814.08,48802.99,81,20.1,80,1013.9,929.9,2026-11-22 15:03:58+00:00


head() and tail() show the first and last 5 rows

## 4.2

In [5]:
rows, cols = weather.shape
print(f"Rows: {rows:,} | Columns: {cols}")
print(weather.columns.tolist())

Rows: 73,060 | Columns: 24
['city', 'country', 'latitude', 'longitude', 'date', 'weather_code', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'sunshine_duration', 'daylight_duration', 'relative_humidity_2m_mean', 'dew_point_2m_mean', 'cloud_cover_mean', 'pressure_msl_mean', 'surface_pressure_mean', 'collected_at']


shape returns a (rows, columns) pair,

## 4.3

In [6]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73060 entries, 0 to 73059
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   city                         73060 non-null  object             
 1   country                      73060 non-null  object             
 2   latitude                     73060 non-null  float64            
 3   longitude                    73060 non-null  float64            
 4   date                         73060 non-null  datetime64[ns]     
 5   weather_code                 73060 non-null  int64              
 6   temperature_2m_max           73060 non-null  float64            
 7   temperature_2m_min           73060 non-null  float64            
 8   temperature_2m_mean          73060 non-null  float64            
 9   precipitation_sum            73060 non-null  float64            
 10  rain_sum                     73060 non-null  f

info() prints each column's name, non-null count, and data type, plus total memory

## 4.4

In [7]:
location_cols    = ["city", "country", "latitude", "longitude"]
date_cols        = ["date"]
categorical_cols = ["city", "country", "weather_code"]
boolean_cols     = []
numeric_cols = [c for c in weather.columns
                if c not in location_cols + date_cols + ["weather_code", "collected_at"]]

print("Location   :", location_cols)
print("Date       :", date_cols)
print("Categorical:", categorical_cols)
print("Boolean    :", boolean_cols, "(none yet; the rainy-day target will be created later)")
print("Numerical  :", numeric_cols)

Location   : ['city', 'country', 'latitude', 'longitude']
Date       : ['date']
Categorical: ['city', 'country', 'weather_code']
Boolean    : [] (none yet; the rainy-day target will be created later)
Numerical  : ['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'sunshine_duration', 'daylight_duration', 'relative_humidity_2m_mean', 'dew_point_2m_mean', 'cloud_cover_mean', 'pressure_msl_mean', 'surface_pressure_mean']


Identifying numerical, categorical, Boolean, date-related, and location variables.collected_at is not a weather feature, so it's excluded from analysis and modeling

## 4.5

In [8]:
weather[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
temperature_2m_max,73060.0,22.494370,10.121488,-26.20,16.6000,24.200,29.7000,47.30
temperature_2m_min,73060.0,13.570739,9.414030,-31.10,7.8000,14.100,21.2000,34.40
temperature_2m_mean,73060.0,17.802529,9.482271,-28.50,12.3000,18.700,25.5000,39.70
precipitation_sum,73060.0,2.658535,6.704097,0.00,0.0000,0.100,2.2000,304.90
rain_sum,73060.0,2.559214,6.637615,0.00,0.0000,0.000,1.9000,304.90
precipitation_hours,73060.0,3.902833,5.531370,0.00,0.0000,1.000,7.0000,24.00
wind_speed_10m_max,73060.0,19.019704,7.401118,3.40,13.6000,17.900,23.0000,95.90
wind_gusts_10m_max,73060.0,38.094656,14.748277,7.20,28.4000,35.600,43.9000,190.10
wind_direction_10m_dominant,73060.0,192.489666,105.191564,0.00,103.0000,208.000,282.0000,360.00
shortwave_radiation_sum,73060.0,16.196337,7.345812,0.01,11.1200,17.030,21.6300,34.53


describe() gives count, mean, std, min, quartiles (25%, 50%, 75%) and max for each numerical column

## 4.6

In [9]:
for col in categorical_cols:
    print(f"\n{col}: {weather[col].nunique()} unique values")
    print(weather[col].value_counts())


city: 20 unique values
city
Lahore         3653
Karachi        3653
Dubai          3653
Delhi          3653
Tokyo          3653
Singapore      3653
Beijing        3653
Sydney         3653
Cairo          3653
Nairobi        3653
Lagos          3653
Cape Town      3653
London         3653
Paris          3653
Moscow         3653
Reykjavik      3653
New York       3653
Toronto        3653
Mexico City    3653
Sao Paulo      3653
Name: count, dtype: int64

country: 19 unique values
country
Pakistan        7306
UAE             3653
India           3653
Japan           3653
Singapore       3653
China           3653
Australia       3653
Egypt           3653
Kenya           3653
Nigeria         3653
South Africa    3653
UK              3653
France          3653
Russia          3653
Iceland         3653
USA             3653
Canada          3653
Mexico          3653
Brazil          3653
Name: count, dtype: int64

weather_code: 13 unique values
weather_code
3     22190
51    13684
0      6753
63  

Identifying unique values and frequency distributions for categorical variables.
nunique() counts distinct values, and value_counts() counts how many rows each value has

## 4.7

In [10]:
mem_mb = weather.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage: {mem_mb:.2f} MB")

Memory usage: 20.00 MB


memory_usage(deep=True) measures bytes per column .sum() adds them up and dividing by 1024**2 converts bytes to megabytes.

## 4.8

In [11]:
missing = weather.isnull().sum()
missing_report = pd.DataFrame({"missing_count": missing,"missing_percent": (missing / len(weather) * 100).round(2)})
missing_report

,missing_count,missing_percent
city,0,0.0
country,0,0.0
latitude,0,0.0
longitude,0,0.0
date,0,0.0
weather_code,0,0.0
temperature_2m_max,0,0.0
temperature_2m_min,0,0.0
temperature_2m_mean,0,0.0
precipitation_sum,0,0.0


In [12]:
missing_by_city = weather.isnull().groupby(weather["city"]).sum().sum(axis=1)
missing_by_year = weather.isnull().groupby(weather["date"].dt.year).sum().sum(axis=1)
print(missing_by_city[missing_by_city > 0])
print(missing_by_year[missing_by_year > 0])

Series([], dtype: int64)
Series([], dtype: int64)


We need to find missing values in each column and inspect missingness by city and date.
Here
->isnull() creates a True/False table and .sum() counts the Trues per column.
->groupby(weather["city"]) splits the True/False table by city
weather["date"].dt.year extracts the year from a datetime column.

## 4.9

In [13]:
print("Fully duplicated rows:", weather.duplicated().sum())
print("Repeated city-date pairs:", weather.duplicated(subset=["city", "date"]).sum())

Fully duplicated rows: 0
Repeated city-date pairs: 0


# 5. Data Cleaning and Preprocessing

## A. Missing values and duplicates

## 1.

Cell 4.8 (the missing_report table) which showed 0 missing values and 0.0% for all 23 columns

## 2.

In [14]:
df = weather.sort_values(["city", "date"]).reset_index(drop=True)

df["is_rainy"] = (df["precipitation_sum"] > 0).astype(int)

df["target_tmax_next"] = df.groupby("city")["temperature_2m_max"].shift(-1)

lag_cols = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
            "precipitation_sum", "wind_speed_10m_max", "relative_humidity_2m_mean",
            "dew_point_2m_mean", "cloud_cover_mean", "pressure_msl_mean",
            "shortwave_radiation_sum"]
for c in lag_cols:
    df[f"{c}_lag1"] = df.groupby("city")[c].shift(1)

before = len(df)
lag_names = [f"{c}_lag1" for c in lag_cols]
df = df.dropna(subset=["target_tmax_next"] + lag_names).reset_index(drop=True)
print(f"Rows before: {before:,} | after: {len(df):,} | dropped: {before - len(df)}")

print(df["is_rainy"].value_counts())
print(df["is_rainy"].value_counts(normalize=True).round(3))

Rows before: 73,060 | after: 73,020 | dropped: 40
is_rainy
1    37357
0    35663
Name: count, dtype: int64
is_rainy
1    0.512
0    0.488
Name: proportion, dtype: float64


The models will predict the next-day maximum temperature and whether a day is rainy, using the previous-day features.I sorted the data by city and date, then created the classification target 'is_rainy' (1 if precipitation is above zero) and the regression target 'target_tmax_next'Rows without a valid previous-day or next-day value were dropped

## 3. and 4.

In [15]:
df_missing = df.copy()

same_day_cols = ["temperature_2m_min", "temperature_2m_mean", "wind_speed_10m_max",
                 "wind_gusts_10m_max", "wind_direction_10m_dominant",
                 "shortwave_radiation_sum", "sunshine_duration",
                 "relative_humidity_2m_mean", "dew_point_2m_mean",
                 "cloud_cover_mean", "pressure_msl_mean", "surface_pressure_mean"]
missing_cols = same_day_cols + lag_names

df_missing[missing_cols] = df_missing[missing_cols].astype(float)

rng = np.random.default_rng(SEED)
for col in missing_cols:
    mask = rng.random(len(df_missing)) < 0.05
    df_missing.loc[mask, col] = np.nan

report = pd.DataFrame({
    "missing_count": df_missing[missing_cols].isnull().sum(),
    "missing_percent": (df_missing[missing_cols].isnull().mean() * 100).round(2)
})
print(report)
#5A-4
protected = ["is_rainy", "target_tmax_next", "temperature_2m_max", "precipitation_sum"]
print("\nMissing values in protected columns:")
print(df_missing[protected].isnull().sum())

df_missing.to_csv(PROCESSED / "weather_with_missing.csv", index=False)

                                missing_count  missing_percent
temperature_2m_min                       3611             4.95
temperature_2m_mean                      3728             5.11
wind_speed_10m_max                       3736             5.12
wind_gusts_10m_max                       3665             5.02
wind_direction_10m_dominant              3610             4.94
shortwave_radiation_sum                  3690             5.05
sunshine_duration                        3709             5.08
relative_humidity_2m_mean                3598             4.93
dew_point_2m_mean                        3673             5.03
cloud_cover_mean                         3677             5.04
pressure_msl_mean                        3769             5.16
surface_pressure_mean                    3663             5.02
temperature_2m_max_lag1                  3654             5.00
temperature_2m_min_lag1                  3645             4.99
temperature_2m_mean_lag1                 3690          

The real data has no missing values, so I made a separate copy, `df_missing`, and randomly blanked about 5% of the cells in 22 input-feature columns, using SEED = 15 so the result is reproducible.

## 6.

Duplicate records (0) and repeated combinations of city-date were identified in section 4.9

### 7.

In [16]:
print("rain_sum > precipitation_sum :", (weather["rain_sum"] > weather["precipitation_sum"] + 1e-6).sum())
print("sunshine > daylight          :", (weather["sunshine_duration"] > weather["daylight_duration"]).sum())
print("temp ordering violated       :", ((weather["temperature_2m_min"] > weather["temperature_2m_mean"]) |
                                         (weather["temperature_2m_mean"] > weather["temperature_2m_max"])).sum())

print("negative precipitation       :", (weather[["precipitation_sum","rain_sum","precipitation_hours"]] < 0).sum().sum())
print("humidity outside 0-100       :", (~weather["relative_humidity_2m_mean"].between(0, 100)).sum())
print("cloud cover outside 0-100    :", (~weather["cloud_cover_mean"].between(0, 100)).sum())
print("wind direction outside 0-360 :", (~weather["wind_direction_10m_dominant"].between(0, 360)).sum())
print("precip hours above 24        :", (weather["precipitation_hours"] > 24).sum())

gaps = weather.sort_values(["city", "date"]).groupby("city")["date"].diff().dt.days
print("max gap between consecutive dates:", gaps.max())

print(weather.groupby("city")[["country", "latitude", "longitude"]].nunique().max())

rain_sum > precipitation_sum : 0
sunshine > daylight          : 0
temp ordering violated       : 0
negative precipitation       : 0
humidity outside 0-100       : 0
cloud cover outside 0-100    : 0
wind direction outside 0-360 : 0
precip hours above 24        : 0
max gap between consecutive dates: 1.0
country      1
latitude     1
longitude    1
dtype: int64


Checked the data for impossible or inconsistent values, date gaps, and mismatched city identifiers.

-> The data has no nulls, but values can still be physically impossible or dates can be skipped, which would corrupt the lag features and models.

# 10. Train-Test Split and Validation

Section 10 says to fit imputation, encoding and scaling using training data only. If we imputed on all the data, the imputer would learn averages from the test period , which is a form of leakage . So the split has to exist before we impute

## 10.1 and 10.2

In [17]:
df_missing["row_in_city"] = df_missing.groupby("city").cumcount()
city_size = df_missing.groupby("city")["date"].transform("size")
is_train = df_missing["row_in_city"] < city_size * 0.8

train = df_missing[is_train].drop(columns="row_in_city").copy()
test  = df_missing[~is_train].drop(columns="row_in_city").copy()

train_complete = df.loc[train.index].copy()
test_complete  = df.loc[test.index].copy()

print(f"Train: {len(train):,} rows | Test: {len(test):,} rows")
print("Train dates:", train["date"].min().date(), "to", train["date"].max().date())
print("Test dates :", test["date"].min().date(), "to", test["date"].max().date())
print("Rows per city in train:", train.groupby("city").size().unique())
print("Rows per city in test :", test.groupby("city").size().unique())

Train: 58,420 rows | Test: 14,600 rows
Train dates: 2016-01-02 to 2023-12-31
Test dates : 2024-01-01 to 2025-12-30
Rows per city in train: [2921]
Rows per city in test : [730]


Because weather data is a time series, I split it chronologically instead of randomly, so the models are never trained on future days. Within each city, the earliest 80% of days went to the training set and the latest 20% to the test set. Every city has the same split sizes, and the test set is kept aside until the final model evaluation.

# 5A-5 Imputation

In [18]:
fill_cols = missing_cols
skew = train[fill_cols].skew()
use_median = skew.abs() >= 0.5

def circular_mean(s):
    rad = np.deg2rad(s.dropna())
    return np.rad2deg(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())) % 360

fill_values = {}
method = {}
for col in fill_cols:
    g = train.groupby("city")[col]
    if col == "wind_direction_10m_dominant":
        fill_values[col], method[col] = g.apply(circular_mean), "circular mean"
    elif use_median[col]:
        fill_values[col], method[col] = g.median(), "median"
    else:
        fill_values[col], method[col] = g.mean(), "mean"

train_imp, test_imp = train.copy(), test.copy()
for col in fill_cols:
    for d in (train_imp, test_imp):
        d[col] = d[col].fillna(d["city"].map(fill_values[col]))

summary = pd.DataFrame({
    "skewness": skew.round(2),
    "method": pd.Series(method),
    "missing_before_train": train[fill_cols].isnull().sum(),
    "missing_after_train": train_imp[fill_cols].isnull().sum(),
    "missing_before_test": test[fill_cols].isnull().sum(),
    "missing_after_test": test_imp[fill_cols].isnull().sum(),
})
print(summary)
print("\nTotal missing before:", train[fill_cols].isnull().sum().sum() + test[fill_cols].isnull().sum().sum())
print("Total missing after :", train_imp[fill_cols].isnull().sum().sum() + test_imp[fill_cols].isnull().sum().sum())

                                skewness         method  missing_before_train  \
temperature_2m_min                 -0.49           mean                  2905   
temperature_2m_mean                -0.59         median                  2999   
wind_speed_10m_max                  1.08         median                  2990   
wind_gusts_10m_max                  1.85         median                  2917   
wind_direction_10m_dominant        -0.21  circular mean                  2869   
shortwave_radiation_sum            -0.28           mean                  2975   
sunshine_duration                  -1.07         median                  2972   
relative_humidity_2m_mean          -0.89         median                  2883   
dew_point_2m_mean                  -0.66         median                  2915   
cloud_cover_mean                   -0.19           mean                  2935   
pressure_msl_mean                  -0.42           mean                  3001   
surface_pressure_mean       

I filled the missing values using training data only, separately for each city, and applied the same values to the test set. Symmetric variables were filled with the mean, skewed ones with the median, and wind direction with a circular mean. Missing values went from 80,531 before imputation to 0 after.

# B.  Categorical encoding and numerical scaling

## 1. to 3.

In [19]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler

north = {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
         6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",11:"Autumn"}
for d in (train_imp, test_imp):
    d["year"] = d["date"].dt.year
    d["month"] = d["date"].dt.month
    d["day_of_year"] = d["date"].dt.dayofyear
    d["season"] = np.where(d["latitude"] >= 0,
                           d["month"].map(north),
                           ((d["month"] + 5) % 12 + 1).map(north))

cat_cols = ["city", "season"]
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train_imp[cat_cols])
train_ohe = pd.DataFrame(ohe.transform(train_imp[cat_cols]),
                         columns=ohe.get_feature_names_out(cat_cols), index=train_imp.index)
test_ohe = pd.DataFrame(ohe.transform(test_imp[cat_cols]),
                        columns=ohe.get_feature_names_out(cat_cols), index=test_imp.index)
print("One-hot columns:", train_ohe.shape[1])

num_cols = ["temperature_2m_min", "temperature_2m_mean", "wind_speed_10m_max",
            "wind_gusts_10m_max", "shortwave_radiation_sum", "sunshine_duration",
            "relative_humidity_2m_mean", "dew_point_2m_mean", "cloud_cover_mean",
            "pressure_msl_mean", "surface_pressure_mean"]

mm = MinMaxScaler().fit(train_imp[num_cols])
ss = StandardScaler().fit(train_imp[num_cols])
train_mm = pd.DataFrame(mm.transform(train_imp[num_cols]), columns=num_cols, index=train_imp.index)
test_mm  = pd.DataFrame(mm.transform(test_imp[num_cols]),  columns=num_cols, index=test_imp.index)
train_ss = pd.DataFrame(ss.transform(train_imp[num_cols]), columns=num_cols, index=train_imp.index)
test_ss  = pd.DataFrame(ss.transform(test_imp[num_cols]),  columns=num_cols, index=test_imp.index)

print("\nTrain, Min-Max (min / max):")
print(train_mm.agg(["min", "max"]).T.round(2).head(5))
print("\nTrain, Z-score (mean / std):")
print(train_ss.agg(["mean", "std"]).T.round(2).head(5))
print("\nTest, Min-Max (min / max):")
print(test_mm.agg(["min", "max"]).T.round(2).head(5))

One-hot columns: 24

Train, Min-Max (min / max):
                         min  max
temperature_2m_min       0.0  1.0
temperature_2m_mean      0.0  1.0
wind_speed_10m_max       0.0  1.0
wind_gusts_10m_max       0.0  1.0
shortwave_radiation_sum  0.0  1.0

Train, Z-score (mean / std):
                         mean  std
temperature_2m_min        0.0  1.0
temperature_2m_mean      -0.0  1.0
wind_speed_10m_max       -0.0  1.0
wind_gusts_10m_max        0.0  1.0
shortwave_radiation_sum  -0.0  1.0

Test, Min-Max (min / max):
                          min   max
temperature_2m_min       0.00  1.00
temperature_2m_mean      0.01  1.00
wind_speed_10m_max      -0.01  0.70
wind_gusts_10m_max      -0.01  0.72
shortwave_radiation_sum  0.00  0.98


Models need numerical input so I one-hot encoded `city` and `season` (24 columns). I dropped `country` because it duplicates `city` info.Applied Min-Max normalization and Z-score standardization to the numerical features, with the encoder and both scalers fitted on training data only and applied to the test set without refitting. Scaling matters for distance-based and gradient-based models such as KNN and logistic regression, while tree-based models do not need it.